# Fotogrametri — automatisk pipeline (Colab)

Henter **koden fra GitHub** og **bildene fra Google Drive**, og kjører hele
pipelinen automatisk.

**Slik kjører du:** Rediger → Notatbokinnstillinger → GPU (T4), deretter
Kjøretid → Kjør alle. Eneste manuelle steg er å godkjenne Drive-tilgang
når popup-en dukker opp.

Forventet i Google Drive (Min disk): `images.zip` med alle bildene.

In [ ]:
# === Konfigurasjon ===
GITHUB_REPO = 'https://github.com/davgei/fotogrametri.git'
DRIVE_ZIP   = '/content/drive/MyDrive/images.zip'  # bildene (zip) i Min disk
MAX_SIZE    = 1600   # lengste bildekant i piksler etter nedskalering
MATCHING    = 'sequential'   # 'sequential' (strekning/runde) eller 'exhaustive'
OVERLAP     = 10
RUN_DENSE   = True   # dense MVS (krever GPU). False = bare sparse

In [ ]:
# 1) Installer avhengigheter
!pip install pycolmap open3d plotly -q

import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() if gpu.returncode == 0 else 'INGEN — sett RUN_DENSE=False')

In [ ]:
# 2) Hent koden fra GitHub
import os, shutil

REPO_DIR = '/content/fotogrametri'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 {GITHUB_REPO} {REPO_DIR}
os.chdir(REPO_DIR)
print('Arbeidsmappe:', os.getcwd())

In [ ]:
# 3) Monter Drive og pakk ut bildene
from google.colab import drive
drive.mount('/content/drive')

import zipfile
from pathlib import Path

RAW_DIR = Path('/content/images_raw')
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True)

assert os.path.exists(DRIVE_ZIP), f'Fant ikke {DRIVE_ZIP} — sjekk DRIVE_ZIP i konfig.'
with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall(RAW_DIR)

# Hvis zip-en pakker ut til en undermappe, flat ut til RAW_DIR
exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
imgs = [p for p in RAW_DIR.rglob('*') if p.suffix.lower() in exts]
for p in imgs:
    if p.parent != RAW_DIR:
        p.rename(RAW_DIR / p.name)
imgs = [p for p in RAW_DIR.iterdir() if p.suffix.lower() in exts]
print(f'Pakket ut {len(imgs)} bilder.')

In [ ]:
# 4) Nedskaler bildene (bruker src/downscale.py fra git)
!python -m src.downscale --input /content/images_raw --output /content/images_small --max_size {MAX_SIZE}

In [ ]:
# 5) Kjør pipelinen (bruker src/reconstruct.py fra git)
dense_flag = '' if RUN_DENSE else '--no_dense'
!python -m src.reconstruct \
    --image_dir /content/images_small \
    --output_dir /content/output \
    --matching {MATCHING} --overlap {OVERLAP} {dense_flag}

## Visualisering

In [ ]:
import numpy as np
import plotly.graph_objects as go
import open3d as o3d
import pycolmap
from pathlib import Path

OUTPUT_DIR = Path('/content/output')
fused_ply  = OUTPUT_DIR / 'dense' / 'fused.ply'

if RUN_DENSE and fused_ply.exists():
    pcd = o3d.io.read_point_cloud(str(fused_ply))
    xyz = np.asarray(pcd.points)
    rgb = np.asarray(pcd.colors)
    label = 'Dense punktsky (MVS)'
else:
    sparse_dir = OUTPUT_DIR / 'sparse'
    model_dirs = [d for d in sparse_dir.iterdir() if d.is_dir()]
    rec = max((pycolmap.Reconstruction(str(d)) for d in model_dirs),
              key=lambda r: r.num_reg_images())
    pts = rec.points3D
    xyz = np.array([p.xyz for p in pts.values()])
    rgb = np.array([p.color for p in pts.values()]) / 255.0
    label = 'Sparse punktsky (SfM)'

MAX_DISPLAY = 200_000
if len(xyz) > MAX_DISPLAY:
    idx = np.random.choice(len(xyz), MAX_DISPLAY, replace=False)
    xyz, rgb = xyz[idx], rgb[idx]

colors = [f'rgb({int(r*255)},{int(g*255)},{int(b*255)})' for r, g, b in rgb]
fig = go.Figure(go.Scatter3d(
    x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
    mode='markers', marker=dict(size=1, color=colors, opacity=0.7)))
fig.update_layout(
    title=f'{label} — {len(xyz):,} punkter',
    scene=dict(aspectmode='data'), height=700,
    margin=dict(l=0, r=0, b=0, t=40))
fig.show()

In [ ]:
# 6) Lagre resultatet tilbake til Drive
DRIVE_OUT = Path('/content/drive/MyDrive/fotogrametri_output')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
if RUN_DENSE and fused_ply.exists():
    shutil.copy2(fused_ply, DRIVE_OUT / 'fused.ply')
    print(f'Lagret {DRIVE_OUT / "fused.ply"}')
else:
    sparse_ply = OUTPUT_DIR / 'sparse_pointcloud.ply'
    rec.export_PLY(str(sparse_ply))
    shutil.copy2(sparse_ply, DRIVE_OUT / 'sparse_pointcloud.ply')
    print(f'Lagret {DRIVE_OUT / "sparse_pointcloud.ply"}')